# 🚀 Notebook 11: Inference Optimization

KV-cache, quantization (4-bit/8-bit), and speculative decoding for fast LLM inference on Apple Silicon.

**Prerequisites:** Notebooks 01-10

In [ ]:
from utils.checks import validate_environment, print_environment_report
env = print_environment_report()

## Inference Bottleneck: Memory Bandwidth

LLM inference is **memory-bandwidth-bound**, not compute-bound. Each token requires reading the entire model from memory:

$$\text{tokens/sec} \approx \frac{\text{bandwidth (GB/s)}}{\text{model size (GB)}}$$

M4 Pro: ~273 GB/s bandwidth. A 7B float16 model (14 GB) → ~19 tok/s theoretical max.

## KV-Cache: Prefill + Decode Pipeline

During autoregressive generation, naive inference recomputes attention for **all** previous tokens at each step → O(n²) total compute.

The **KV-cache** stores K and V projections from previous steps:
- **Prefill phase**: Process the full prompt in one forward pass, populate the cache
- **Decode phase**: Generate one token at a time, reading cached K/V → O(1) per new token

$\text{Without cache: } O(n^2 \cdot d) \quad \text{With cache: } O(n \cdot d) \text{ total}$

💡 KV-cache trades **memory** for **compute** — for a 7B model at seq_len=2048, the cache is ~1GB in float16.

🎯 **Interview tip**: The prefill phase is compute-bound (matrix multiplications), while the decode phase is memory-bandwidth-bound (reading model weights for each token).

In [ ]:
from utils.inference import SimpleLM, KVCacheManager
import mlx.core as mx
import numpy as np

# Create a small language model
mx.random.seed(42)
model = SimpleLM(vocab_size=64, d_model=32, n_layers=2, d_ff=64)
mx.eval(model.parameters())

# Simulate a prompt
prompt_ids = mx.array([[1, 5, 12, 8, 3]])  # [1, 5]
gen_tokens = 4

# --- Generate with KV-cache ---
cache_mgr = KVCacheManager(model)
generated_ids, cached_logits = cache_mgr.generate(prompt_ids, gen_tokens)
mx.eval(generated_ids)

# --- Full recomputation (no cache) for verification ---
full_ids = mx.concatenate([prompt_ids, generated_ids], axis=1)
full_logits = KVCacheManager.generate_without_cache(model, full_ids)

# Verify equivalence at each generated position
prompt_len = prompt_ids.shape[1]
print("🎯 KV-Cache Equivalence Verification:")
for t in range(gen_tokens):
    cached_t = np.array(cached_logits[t][0, 0, :])
    full_t = np.array(full_logits[0, prompt_len - 1 + t, :])
    max_diff = np.max(np.abs(cached_t - full_t))
    print(f"  Position {t}: max logit diff = {max_diff:.2e} ✅" if max_diff < 1e-4
          else f"  Position {t}: max logit diff = {max_diff:.2e} ❌")

print(f"\n💡 Generated tokens: {np.array(generated_ids[0])}")
print(f"⚡ Cache memory: {cache_mgr.memory_bytes() / 1024:.1f} KB")
print(f"   Without cache: O(n²) compute per token")
print(f"   With cache:    O(n) compute per token")

## Quantization: 4-bit and 8-bit Weight Compression

Reduce model size by representing weights with fewer bits using **group-wise min-max quantization**:

$$\text{code} = \text{round}\left(\frac{x - x_{\min}}{(x_{\max} - x_{\min}) / (2^N - 1)}\right), \quad \text{code} \in [0, 2^N - 1]$$

**Error bound per group**: $\max|x - \hat{x}| \leq \frac{x_{\max} - x_{\min}}{2^N - 1}$

| Bit Width | Levels | Compression | Typical Quality Loss |
|-----------|--------|-------------|---------------------|
| 8-bit     | 255    | ~2x         | Negligible          |
| 4-bit     | 15     | ~4x         | Small (~0.1 PPL)    |

💡 **Group-wise** quantization limits error to each group's range — much better than per-tensor.

⚠️ Smaller group sizes give better accuracy but more overhead (more scales to store).

In [ ]:
from utils.inference import Quantizer

# Create random weights (simulating a model layer)
mx.random.seed(42)
w = mx.random.normal((256, 256))
mx.eval(w)

# --- 4-bit quantization ---
q4 = Quantizer.quantize_4bit(w, group_size=64)
w4 = Quantizer.dequantize(q4)
stats4 = Quantizer.compression_stats(w, q4)

# --- 8-bit quantization ---
q8 = Quantizer.quantize_8bit(w, group_size=64)
w8 = Quantizer.dequantize(q8)
stats8 = Quantizer.compression_stats(w, q8)

print("⚡ Quantization Comparison:")
print(f"  Original:  {stats4['original_bytes']/1024:.1f} KB")
print(f"  8-bit:     {stats8['quantized_bytes']/1024:.1f} KB  "
      f"({stats8['compression_ratio']:.1f}x compression, max error: {stats8['max_error']:.6f})")
print(f"  4-bit:     {stats4['quantized_bytes']/1024:.1f} KB  "
      f"({stats4['compression_ratio']:.1f}x compression, max error: {stats4['max_error']:.6f})")

# Verify error bound
bounds_4 = Quantizer.compute_error_bound(w, bits=4, group_size=64)
bounds_8 = Quantizer.compute_error_bound(w, bits=8, group_size=64)
print(f"\n🎯 Theoretical max error bound (4-bit): {mx.max(bounds_4).item():.6f}")
print(f"   Actual max error (4-bit):             {stats4['max_error']:.6f}")
print(f"   Theoretical max error bound (8-bit):  {mx.max(bounds_8).item():.6f}")
print(f"   Actual max error (8-bit):             {stats8['max_error']:.6f}")
print(f"\n💡 Error is always ≤ theoretical bound — guaranteed by min-max quantization!")

## MLX Built-in Quantization

MLX provides `nn.QuantizedLinear` for efficient quantized inference — the same approach used by `mlx-lm` when loading 4-bit models from Hugging Face.

⚡ `mlx-lm` uses these for loading GGUF and safetensors quantized models:
- `mx.quantize()` — quantize a tensor to N-bit
- `mx.dequantize()` — reconstruct from quantized representation
- `nn.QuantizedLinear` — drop-in replacement for `nn.Linear` with quantized weights

In [ ]:
# MLX built-in quantization demo
print("MLX quantization support:")
print("  mx.quantize()        — quantize a tensor")
print("  mx.dequantize()      — dequantize back")
print("  nn.QuantizedLinear   — quantized linear layer")
print()

# Compare our implementation with MLX built-in
w_test = mx.random.normal((128, 128))
mx.eval(w_test)

# Our 4-bit
q_ours = Quantizer.quantize_4bit(w_test, group_size=64)
w_ours = Quantizer.dequantize(q_ours)
our_error = mx.max(mx.abs(w_test - w_ours)).item()

# MLX built-in 4-bit
q_mlx, scales_mlx, biases_mlx = mx.quantize(w_test, bits=4, group_size=64)
w_mlx = mx.dequantize(q_mlx, scales_mlx, biases_mlx, bits=4, group_size=64)
mlx_error = mx.max(mx.abs(w_test - w_mlx)).item()

print(f"⚡ Our 4-bit max error:  {our_error:.6f}")
print(f"   MLX 4-bit max error:  {mlx_error:.6f}")
print(f"\n💡 Both use the same min-max quantization principle!")

## Speculative Decoding: Draft + Verify Pipeline

Use a small **draft model** to generate N candidate tokens cheaply, then **verify** them in parallel with the large target model. If the draft model is good, most tokens are accepted → speedup!

**Algorithm:**
1. Draft model generates N tokens autoregressively (fast, small model)
2. Target model processes [prompt + N draft tokens] in ONE forward pass
3. Compare: accept tokens where draft matches target's greedy choice
4. On first mismatch: use target's token instead, discard remaining drafts

🎯 **Key insight**: accepted tokens are **identical** to what the target model would generate alone — speculative decoding is a pure speed optimization with **zero quality loss** (for greedy decoding).

⚡ Typical speedup: 2-3x with a well-matched draft model.

⚠️ Acceptance rate depends on how well the draft model approximates the target.

In [ ]:
from utils.inference import SpeculativeDecoder, create_model_pair
import time

# Create draft (small) and target (large) models
mx.random.seed(42)
draft_model, target_model = create_model_pair(
    vocab_size=64,
    d_model_draft=16, d_model_target=32,
    n_layers_draft=1, n_layers_target=2,
    d_ff_draft=32, d_ff_target=64,
)

prompt = mx.array([[1, 5, 12, 8, 3]])
n_generate = 6
n_draft = 3

# --- Speculative decoding ---
decoder = SpeculativeDecoder(draft_model, target_model)
spec_tokens, stats = decoder.generate(prompt, max_new_tokens=n_generate, n_draft=n_draft)
mx.eval(spec_tokens)

# --- Target-only reference ---
ref_tokens = SpeculativeDecoder.generate_target_only(target_model, prompt, n_generate)
mx.eval(ref_tokens)

print("🎯 Speculative Decoding Results:")
print(f"  Draft tokens per iteration: {n_draft}")
print(f"  Acceptance rate: {stats['acceptance_rate']:.1%}")
print(f"  Speculative output: {np.array(spec_tokens[0])}")
print(f"  Target-only output: {np.array(ref_tokens[0])}")

# Verify correctness: accepted tokens match target-only
n_check = min(spec_tokens.shape[1], ref_tokens.shape[1])
match = np.array_equal(np.array(spec_tokens[0, :n_check]), np.array(ref_tokens[0, :n_check]))
print(f"\n  Tokens match target-only: {'✅ Yes' if match else '❌ No'}")
print(f"\n⚡ Speedup comes from verifying {n_draft} tokens in ONE target forward pass")
print(f"   instead of {n_draft} separate forward passes.")

## Quantization Formats: GPTQ, AWQ, GGUF

Modern LLM quantization goes beyond simple min-max. Here are the three dominant formats:

### GPTQ (2022) — Post-Training Quantization via Optimal Brain Quantization

GPTQ quantizes weights one column at a time, using the **Hessian** (second-order information from calibration data) to minimize the output error:

$$\hat{w}_i = \text{argmin}_{q} \frac{(w_i - q)^2}{[H^{-1}]_{ii}}$$

- Quantizes to 4-bit or 3-bit with group sizes of 32-128
- Requires a small calibration dataset (~128 samples)
- 🎯 **Interview tip**: GPTQ is "optimal" in the sense that it minimizes layer-wise reconstruction error using second-order information

### AWQ (2023) — Activation-Aware Weight Quantization

AWQ observes that **not all weights are equally important** — weights connected to high-magnitude activations matter more:

1. Identify "salient" weight channels (those with large activation magnitudes)
2. Scale salient channels up before quantization, scale down after
3. This preserves important weights at higher precision within the same bit budget

- No retraining needed, just calibration data
- ⚡ Often better quality than GPTQ at the same bit width

### GGUF (2023) — GPT-Generated Unified Format

GGUF is a **file format** (not a quantization algorithm) designed for efficient local inference:

- Successor to GGML format, used by `llama.cpp`
- Supports multiple quantization types: Q4_0, Q4_K_M, Q5_K_M, Q8_0, etc.
- "K" variants use k-quant: different bit widths for different layers based on importance
- Self-contained: model weights + tokenizer + metadata in one file
- ⚡ Optimized for CPU inference with SIMD, also supports Metal GPU

| Format | Algorithm | Calibration | Best For |
|--------|-----------|-------------|----------|
| GPTQ   | Hessian-based OBQ | Yes (128 samples) | GPU inference |
| AWQ    | Activation-aware scaling | Yes (calibration) | GPU inference |
| GGUF   | Various (Q4_K_M, etc.) | No (some variants) | CPU/Metal local inference |

⚠️ All three formats can achieve 4-bit quantization with <1 perplexity point increase on most models.

## 📊 Visualizing the Quantization Speed Payoff

Since inference is memory-bandwidth-bound, **smaller model = fewer bytes to read from memory = faster token generation**.

The formula is simple: $\text{tokens/sec} = \frac{\text{bandwidth (GB/s)}}{\text{model size (GB)}}$

Let's compute the theoretical maximum tokens/sec for a 7B model at different quantization levels on an M4 Pro (273 GB/s memory bandwidth).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# M4 Pro memory bandwidth
bandwidth_gb_s = 273  # GB/s

# 7B model sizes at different quantization levels
# 7B params × bytes_per_param
quant_levels = ['float32\n(28 GB)', 'float16\n(14 GB)', '8-bit\n(7 GB)', '4-bit\n(3.5 GB)']
model_sizes_gb = [28.0, 14.0, 7.0, 3.5]  # 7B × 4, 2, 1, 0.5 bytes

# Theoretical max: tokens/sec = bandwidth / model_size
tokens_per_sec = [bandwidth_gb_s / size for size in model_sizes_gb]

# --- Bar chart ---
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4']  # red → blue (slow → fast)
bars = ax.bar(quant_levels, tokens_per_sec, color=colors, edgecolor='black', linewidth=0.8)

# Add value labels on bars
for bar, tps in zip(bars, tokens_per_sec):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{tps:.0f} tok/s', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Theoretical Max Tokens/sec', fontsize=12)
ax.set_title(f'Why Quantization Speeds Up Inference\n'
             f'7B Model on M4 Pro ({bandwidth_gb_s} GB/s bandwidth)',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, max(tokens_per_sec) * 1.2)
ax.axhline(y=tokens_per_sec[1], color='gray', linestyle='--', alpha=0.4, label='float16 baseline')
ax.legend(fontsize=10)

# Annotation
ax.annotate('4× faster than float32!',
            xy=(3, tokens_per_sec[3]), xytext=(1.8, tokens_per_sec[3] * 0.85),
            fontsize=10, color='#1f77b4', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#1f77b4', lw=1.5))

plt.tight_layout()
plt.savefig('figures/11_quantization_speed.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 Key insight: Smaller model = fewer bytes to read from memory = faster token generation.")
print(f"   At 4-bit, a 7B model fits in just 3.5 GB → {tokens_per_sec[3]:.0f} tok/s theoretical max!")
print(f"   That's {tokens_per_sec[3]/tokens_per_sec[0]:.0f}× faster than float32.")

## 📜 History & Alternatives: Inference Optimization

The evolution of LLM inference optimization tracks the shift from compute-bound to memory-bandwidth-bound workloads:

| Year | Innovation | Key Contribution |
|------|-----------|-----------------|
| ~2017 | **FP32 Training** | Standard full-precision training and inference |
| 2018 | **FP16 / Mixed Precision** (Micikevicius et al., NVIDIA) | 2x speedup with loss scaling for training stability |
| 2020 | **INT8 Quantization** (Zafrir et al.) | Post-training quantization for inference, ~2x compression |
| 2022 | **LLM.int8()** (Dettmers et al.) | Mixed-precision decomposition — outlier features in FP16, rest in INT8 |
| 2022 | **GPTQ** (Frantar et al.) | Hessian-based one-shot 4-bit quantization via Optimal Brain Quantization |
| 2022 | **Speculative Decoding** (Leviathan et al., Chen et al.) | Draft-verify pipeline for 2-3x speedup with no quality loss |
| 2023 | **AWQ** (Lin et al., MIT) | Activation-aware weight quantization — protect salient channels |
| 2023 | **GGML → GGUF** (Gerganov et al.) | Unified format for local inference, powers llama.cpp ecosystem |
| 2023 | **Continuous Batching** (Yu et al., vLLM) | Dynamic request scheduling for higher GPU utilization in serving |
| 2024 | **AQLM** (Egiazarian et al.) | Additive quantization for extreme 2-bit compression |
| 2024 | **QuIP#** (Tseng et al.) | Lattice-based quantization with incoherence processing |
| 2025 | **1-bit LLMs** (Ma et al., BitNet b1.58) | Ternary weights {-1, 0, 1} — replaces multiplications with additions |

**Key trend**: Each generation pushes lower bit widths while maintaining quality. The frontier has moved from FP32 (2017) → FP16 (2018) → INT8 (2020) → INT4 (2022) → INT2 (2024) → 1.58-bit (2025).

🎯 **Interview tip**: Know the tradeoff — lower bits = faster inference + less memory, but calibration and format choice matter. GPTQ uses second-order info, AWQ uses activation awareness, GGUF optimizes for local deployment.

**Next:** Notebook 12 — Flash, Paged, and Ring Attention